# The corpus

Everything downstream — embeddings, search, reranking, the taste vector — reads through one record shape. Get that shape wrong here and every later notebook inherits the mistake, so this notebook does exactly two things: fetch real papers from arXiv's public API, and normalize them into a frozen contract before anything else gets built.

No API key needed for this part — arXiv's API is open. `OPENAI_API_KEY` shows up starting in notebook 2, for embeddings.

**Fetching.** arXiv's API (`export.arxiv.org/api/query`) is an Atom feed, paged 100 entries at a time, sorted by submission date. We walk each category newest-first and stop paging as soon as a page's oldest entry crosses `since` — no need to fetch and discard pages we don't want.

A paper can carry more than one category (`cs.CL` and `cs.AI` both), so fetching category-by-category and merging by id is deliberate: it's how a paper ends up with every category it actually belongs to, not just the one we happened to search for. `primary_category` stays whatever arXiv reports as primary.

The 3-second delay between requests is arXiv's own API etiquette guideline, not a guess — see the [API docs](https://info.arxiv.org/help/api/user-manual.html).

The implementation lives in [`readnext/corpus.py`](readnext/corpus.py) rather than inline here — every later notebook needs `readnext.corpus.load()` importable, starting with notebook 2's embedding cache.

In [1]:
from readnext.config import ARXIV_CATEGORIES, ARXIV_SINCE, ARXIV_TARGET_COUNT, PAPERS_FILE
from readnext.corpus import fetch_corpus, load, save

ARXIV_CATEGORIES, ARXIV_SINCE, ARXIV_TARGET_COUNT

(['cs.CL', 'cs.LG', 'cs.AI', 'cs.IR'], '2023-01-01', 2000)

In [2]:
# ~2,000 papers across 4 categories, at ~3s/page this takes a few minutes.
papers = fetch_corpus(ARXIV_CATEGORIES, ARXIV_SINCE, ARXIV_TARGET_COUNT)
len(papers)

1693

**The record contract.** One paper, normalized. Everything from notebook 2 onward assumes exactly these fields exist and mean this.

In [3]:
papers[0]

Paper(id='2608.23566v1', title='How to Train a Critic Stably and Efficiently', abstract='Group-based reinforcement learning methods such as GRPO for large language models avoid training a critic by sampling multiple responses for each prompt. A reliable critic could instead estimate token-level advantages from one response, but standard critic-based training recipes are often unstable. We study this instability and develop \\textbf{Best-Practice Critic Optimization (BPCO)}, a recipe that combines DPPO, value predictions bounded to the reward range, Monte Carlo value targets, unnormalized policy advantages, and length-adaptive generalized advantage estimation. Because the critic is used only during training, BPCO can also condition it on reward-defining information, such as a reference answer or grading rubric, that is hidden from the policy. Controlled experiments isolate the effect of each design choice. Across mathematical reasoning tasks with models ranging from 1.5B parameters to 3

**Sanity checks before committing the corpus.** Worth confirming before `papers.jsonl` becomes the fixed corpus every result in this project gets compared against: how the categories break down, and that multi-category papers actually merged instead of appearing twice.

In [4]:
from collections import Counter

print("unique ids:", len(papers), "==", len({p.id for p in papers}))
print("multi-category papers:", sum(1 for p in papers if len(p.categories) > 1))
Counter(p.primary_category for p in papers)

unique ids: 1693 == 1693
multi-category papers: 1095


Counter({'cs.CL': 365,
         'cs.IR': 330,
         'cs.LG': 318,
         'cs.AI': 298,
         'cs.CV': 106,
         'cs.CR': 36,
         'cs.RO': 21,
         'cs.SE': 20,
         'stat.ML': 20,
         'cs.HC': 15,
         'cs.SD': 15,
         'cs.DB': 14,
         'stat.ME': 8,
         'cs.CY': 7,
         'eess.AS': 6,
         'cs.MA': 6,
         'quant-ph': 6,
         'cs.SI': 5,
         'cs.DC': 5,
         'cs.NI': 5,
         'eess.SP': 5,
         'eess.IV': 5,
         'cs.AR': 5,
         'math.NA': 4,
         'math.OC': 4,
         'cond-mat.mtrl-sci': 4,
         'astro-ph.IM': 3,
         'physics.plasm-ph': 3,
         'hep-th': 3,
         'cs.IT': 3,
         'cs.GT': 3,
         'cs.DL': 3,
         'cs.NE': 2,
         'cs.DS': 2,
         'eess.SY': 2,
         'stat.CO': 2,
         'econ.GN': 2,
         'cs.MM': 2,
         'cs.LO': 1,
         'astro-ph.EP': 1,
         'q-fin.PM': 1,
         'q-bio.OT': 1,
         'cs.MS': 1,
         'hep-p

**Commit the corpus.** `papers.jsonl` is committed so the corpus — and every recall/MRR/nDCG number computed against it later — stays fixed across runs.

In [5]:
save(papers, PAPERS_FILE)

reloaded = load(PAPERS_FILE)
assert reloaded == papers
PAPERS_FILE, len(reloaded)

(PosixPath('/Users/oleksandr/Documents/Code/python/ai-bootcamp/openai/read-next-project/data/papers.jsonl'),
 1693)

**Token counts and cost.** "2,000 papers" isn't a cost until it's tokens. `tiktoken` counts them the same way the embeddings endpoint will bill them, so this is the number to check before running notebook 2 for real.

In [6]:
from readnext.corpus import count_tokens, estimate_embedding_cost

total_tokens, cost_usd = estimate_embedding_cost([p.abstract for p in papers])
avg_tokens = total_tokens / len(papers)
print(f"{total_tokens:,} tokens across {len(papers)} abstracts (avg {avg_tokens:.0f}/paper)")
print(f"embedding the whole corpus once: ${cost_usd:.4f}")

458,046 tokens across 1693 abstracts (avg 271/paper)
embedding the whole corpus once: $0.0092


**What's a chunk?** The abstracts are short — 150 to 300 words — which makes the chunking decision concrete without a PDF pipeline: embed a narrow window so search can match a specific claim, or embed the whole abstract so nothing needs reassembling later. Three strategies, compared by eye on the same paper.

In [7]:
from readnext.corpus import chunk

sample = papers[:2]
for strategy in ["whole", "window", "small_to_big"]:
    chunks = chunk(sample, strategy)
    print(f"--- {strategy} ({len(chunks)} chunks for {len(sample)} papers) ---")
    for c in chunks[:3]:
        same = "context == text" if c.context == c.text else f"context is the full abstract ({len(c.context)} chars)"
        print(f"  [{c.chunk_id}] {same}")
        print(f"    text: {c.text[:100]}...")
    print()

--- whole (2 chunks for 2 papers) ---
  [2608.23566v1:0] context == text
    text: Group-based reinforcement learning methods such as GRPO for large language models avoid training a c...
  [2608.23564v1:0] context == text
    text: Modern software systems accumulate technical debt over decades of development, which makes migration...

--- window (22 chunks for 2 papers) ---
  [2608.23566v1:0] context == text
    text: Group-based reinforcement learning methods such as GRPO for large language models avoid training a c...
  [2608.23566v1:1] context == text
    text: A reliable critic could instead estimate token-level advantages from one response, but standard crit...
  [2608.23566v1:2] context == text
    text: We study this instability and develop \textbf{Best-Practice Critic Optimization (BPCO)}, a recipe th...

--- small_to_big (22 chunks for 2 papers) ---
  [2608.23566v1:0] context is the full abstract (1333 chars)
    text: Group-based reinforcement learning methods such as GRPO fo

`window` and `small_to_big` embed the identical text — the only difference is what `context` points back to. That's the whole trick: small-to-big costs nothing extra to embed, it just keeps a pointer to the full abstract so search can match on a narrow claim and still hand back something readable.

In [8]:
for strategy in ["whole", "window", "small_to_big"]:
    chunks = chunk(papers, strategy)
    tokens, cost = estimate_embedding_cost([c.text for c in chunks])
    print(f"{strategy:14s} chunks={len(chunks):6,d}  tokens={tokens:8,d}  cost=${cost:.4f}")

whole          chunks= 1,693  tokens= 458,046  cost=$0.0092
window         chunks=12,320  tokens= 812,964  cost=$0.0163
small_to_big   chunks=12,320  tokens= 812,964  cost=$0.0163


**Decision: `small_to_big`.** `whole` is cheapest but a query that matches one sentence of a long abstract still has to compete on the full-abstract vector, which blurs precision; `window` fixes that but returns a fragment with no way back to the paper. `small_to_big` gets the window's precision at the same embedding cost as `window` (see above — the tokens are identical), and its `context` field means search can retrieve on a sentence and still hand back the whole abstract. That's the strategy `readnext.embed` builds the index against in notebook 2.